[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Pruebas de hipótesis y tamaño del efecto.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Martes 27 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se aplicarán pruebas *t* de Student y de Welch, Mann-Whitney, pruebas por permutación, la prueba *z* de proporciones, la chi cuadrada de independencia y el ANOVA de una vía con Levene, Welch y Tukey, siempre acompañadas del tamaño del efecto (d de Cohen, V de Cramér y η²), usando `penguins`, `titanic` y `tips`.

## 🎯 Objetivo de la sesión.

**Elegir la prueba adecuada, revisar sus supuestos e informar el tamaño del efecto junto con el valor *p*.**

Al terminar podrás:
1. Explicar H0/H1, el valor *p*, los errores tipo I y II, la potencia y el riesgo de las comparaciones múltiples.
2. Comparar dos grupos con *t* de Welch, Mann-Whitney o permutación, según los datos.
3. Probar proporciones y la independencia de dos variables categóricas.
4. Comparar más de dos grupos con ANOVA, revisar la igualdad de varianzas y localizar diferencias con Tukey.
5. Reportar d de Cohen, V de Cramér y η², e interpretar su magnitud.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Repaso |
| 20 | 1. La lógica del contraste |
| 25 | 2. Dos grupos |
| 20 | 3. Proporciones y variables categóricas |
| 25 | 4. Más de dos grupos |
| 20 | 5. Tamaño del efecto y ejercicio de la sesión |

🟢 = ejercicio básico · 🔴 = reto opcional

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

---
## 0. Repaso (10 min).

En la sesión 5, un IC del 95% para la diferencia de masa media entre machos y hembras Gentoo fue de unos 700 a 910 g: el 0 quedó muy lejos. Una prueba de hipótesis responde la pregunta complementaria: **si en la población no hubiera diferencia, ¿qué tan raro sería observar una diferencia como esta?** IC y prueba se complementan; el IC además informa la **magnitud**.

---
## 1. La lógica del contraste (20 min).

| Concepto | Significado |
|---|---|
| H0 (hipótesis nula) | "No hay efecto": diferencia 0, independencia, medias iguales |
| H1 (alternativa) | Hay diferencia (bilateral) o va en una dirección (unilateral, decidida **antes** de ver los datos) |
| Valor *p* | Probabilidad, **si H0 fuera cierta**, de obtener un resultado al menos tan extremo como el observado |
| α (nivel de significancia) | Umbral fijado de antemano (habitualmente 0.05) |
| Error tipo I | Rechazar H0 siendo cierta (probabilidad α) |
| Error tipo II | No rechazar H0 siendo falsa (probabilidad β) |
| Potencia | 1 − β: probabilidad de detectar un efecto que sí existe |

**Advertencia:** el valor *p* **no** es la probabilidad de que H0 sea cierta, no mide el tamaño del efecto y *p* > 0.05 **no** demuestra que no haya diferencia.

**Ejemplo: comparaciones múltiples y *p-hacking*.** Se simularán 1,000 "estudios" en los que se comparan 20 variables entre dos grupos **idénticos** (H0 es cierta en todas). ¿En qué fracción de estudios aparece al menos un *p* < 0.05?

* La siguiente celda simulará los 1,000 estudios con 20 pruebas *t* cada uno, de forma vectorizada.

In [ ]:
grupo_a = rng.normal(size=(1000, 20, 30))
grupo_b = rng.normal(size=(1000, 20, 30))
valores_p = stats.ttest_ind(grupo_a, grupo_b, axis=2).pvalue
float((valores_p < 0.05).any(axis=1).mean())

Cerca de 64% de los estudios "encuentra algo" (1 − 0.95²⁰ ≈ 0.64) sin que exista ningún efecto. Probar muchas variables, subgrupos o versiones de un análisis y reportar solo lo significativo es *p-hacking*. **Decisiones:** definir las preguntas antes de mirar los resultados, reportar todas las pruebas hechas y, si son muchas, corregir con Holm o Bonferroni (`multipletests`).

* La siguiente celda aplicará la corrección de Holm a las 20 pruebas de cada estudio y calculará en qué fracción de estudios queda algún hallazgo.

In [ ]:
algun_hallazgo_holm = np.array([multipletests(p_estudio, alpha=0.05, method="holm")[0].any() for p_estudio in valores_p])
float(algun_hallazgo_holm.mean())

Con la corrección, la fracción de estudios con algún falso hallazgo vuelve a quedar alrededor de α = 5%. El costo es menos potencia para cada prueba individual: por eso conviene hacer pocas pruebas, bien elegidas.

### 1.1 Potencia: ¿cuántos datos hacen falta?

La potencia depende del tamaño del efecto, de *n* y de α. `TTestIndPower` de *statsmodels* resuelve cualquiera de ellos dados los demás. El tamaño del efecto se expresa con la **d de Cohen** (diferencia de medias en unidades de desviación estándar).

* La siguiente celda calculará el *n* por grupo necesario para detectar d = 0.5 con 80% de potencia, y la potencia que se tendría con d = 0.3 y 30 por grupo.

In [ ]:
potencia = TTestIndPower()
(round(potencia.solve_power(effect_size=0.5, alpha=0.05, power=0.8), 1),
 round(float(potencia.power(effect_size=0.3, nobs1=30, alpha=0.05)), 3))

Se necesitan unos 64 casos por grupo para un efecto mediano; con 30 por grupo y un efecto pequeño (d = 0.3), solo se detectaría 1 de cada 5 veces. **Consecuencia:** en un estudio pequeño, "no significativo" casi no informa nada.

### 🟢 Ejercicio 1 · ¿Qué dice un valor *p*?

Una prueba compara la masa de dos colonias y obtiene *p* = 0.03. Indica si cada afirmación es correcta: (a) "hay 3% de probabilidad de que H0 sea cierta"; (b) "si las colonias no difirieran, una diferencia así de grande o mayor ocurriría en torno a 3% de las veces"; (c) "la diferencia es grande"; (d) "con *p* = 0.30 se habría demostrado que las colonias son iguales".

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 2. Dos grupos (25 min).

| Prueba | Compara | Supuestos | En *SciPy* |
|---|---|---|---|
| *t* de Student | Medias | Independencia, normalidad aproximada (o *n* grande), **varianzas iguales** | `ttest_ind(a, b)` |
| *t* de Welch | Medias | Independencia, normalidad aproximada; **no** exige varianzas iguales | `ttest_ind(a, b, equal_var=False)` |
| Mann-Whitney | Si un grupo tiende a tener valores mayores (rangos) | Independencia; robusta a atípicos y asimetría | `mannwhitneyu(a, b)` |
| Permutación | Cualquier estadístico (diferencia de medias, de medianas…) | Independencia e intercambiabilidad bajo H0 | `permutation_test((a, b), estadistico)` |

**Decisión práctica:** usar **Welch por defecto** para medias; Mann-Whitney o permutación de medianas si hay asimetría fuerte o atípicos con *n* pequeño. La **independencia** no se arregla con ninguna prueba: mediciones repetidas del mismo individuo requieren pruebas pareadas.

**Consecuencia: ¿Student o Welch?** Se simularán 5,000 comparaciones con H0 cierta (medias iguales), pero con un grupo pequeño y muy variable (*n* = 10, σ = 3) y otro grande y poco variable (*n* = 50, σ = 1). Con α = 0.05, la prueba debería equivocarse 5% de las veces.

* La siguiente celda calculará la tasa real de error tipo I de Student y de Welch en ese escenario.

In [ ]:
pequeno_variable = rng.normal(0, 3, size=(5000, 10))
grande_estable = rng.normal(0, 1, size=(5000, 50))
{"Student": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1).pvalue < 0.05).mean()),
 "Welch": float((stats.ttest_ind(pequeno_variable, grande_estable, axis=1, equal_var=False).pvalue < 0.05).mean())}

Student rechaza H0 en torno a 28% de las veces (¡más de cinco veces α!), mientras que Welch se mantiene en ≈ 5%. Cuando las varianzas son iguales, Welch pierde muy poca potencia; por eso es la opción por defecto.

**Ejemplo:** masa de machos y hembras Adelie. Se revisan los supuestos, se aplican las pruebas y se calcula el tamaño del efecto.

* La siguiente celda definirá `d_cohen`, la diferencia de medias dividida entre la desviación estándar combinada.

In [ ]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

* La siguiente celda separará las masas de machos y hembras Adelie y revisará normalidad (Shapiro-Wilk) e igualdad de varianzas (Levene).

In [ ]:
adelie = pinguinos[pinguinos["species"] == "Adelie"]
machos_adelie = adelie.loc[adelie["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_adelie = adelie.loc[adelie["sex"] == "Female", "body_mass_g"].to_numpy()
{"n": (len(machos_adelie), len(hembras_adelie)),
 "Shapiro machos p": round(float(stats.shapiro(machos_adelie).pvalue), 3),
 "Shapiro hembras p": round(float(stats.shapiro(hembras_adelie).pvalue), 3),
 "Levene p": round(float(stats.levene(machos_adelie, hembras_adelie).pvalue), 3)}

**Nota:** las pruebas de normalidad tienen poca potencia con *n* pequeño y rechazan por detalles irrelevantes con *n* grande. Conviene complementarlas con un histograma o un gráfico Q-Q, y recordar que con *n* ≥ 30 por grupo la prueba *t* es robusta a desviaciones moderadas.

* La siguiente celda aplicará Welch, Mann-Whitney y una prueba por permutación de la diferencia de medias (9,999 permutaciones, `random_state=401`).

In [ ]:
welch = stats.ttest_ind(machos_adelie, hembras_adelie, equal_var=False)
mann_whitney = stats.mannwhitneyu(machos_adelie, hembras_adelie)
permutacion = stats.permutation_test((machos_adelie, hembras_adelie),
                                     lambda a, b, axis=-1: np.mean(a, axis=axis) - np.mean(b, axis=axis),
                                     n_resamples=9999, random_state=401)
pd.DataFrame({
    "estadístico": [welch.statistic, mann_whitney.statistic, permutacion.statistic],
    "valor p": [welch.pvalue, mann_whitney.pvalue, permutacion.pvalue],
}, index=["Welch", "Mann-Whitney", "permutación"])

* La siguiente celda calculará la diferencia de medias con su IC del 95% (Welch) y la d de Cohen.

In [ ]:
ic_welch = welch.confidence_interval(confidence_level=0.95)
(round(float(machos_adelie.mean() - hembras_adelie.mean()), 1), np.round([ic_welch.low, ic_welch.high], 1),
 round(float(d_cohen(machos_adelie, hembras_adelie)), 2))

Las tres pruebas coinciden: *p* minúsculo. El valor *p* de la permutación (0.0002) es el mínimo posible con 9,999 permutaciones en una prueba bilateral: ninguna permutación igualó la diferencia observada, así que el valor real es todavía menor. Lo informativo es la magnitud: los machos pesan unos 675 g más (IC 95%: 573 a 776 g), una d de 2.17, un efecto **muy grande**.

| d de Cohen (orientativo) | 0.2 | 0.5 | 0.8 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

### 🟢 Ejercicio 2 · "No significativo" no es "igual".

Compara con Welch la masa de los Chinstrap y la de los Adelie. Reporta la diferencia de medias, su IC del 95%, el valor *p* y la d de Cohen. ¿Se puede afirmar que pesan lo mismo?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 3 · Propina porcentual de fumadores y no fumadores.

`tip_pct` es asimétrica y tiene atípicos (sesión 3). Compara fumadores (`smoker == "Yes"`) y no fumadores con Mann-Whitney y con una prueba por permutación de la **diferencia de medianas** (`random_state=401`). Reporta las medianas de cada grupo.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Verifica la potencia por simulación.

`TTestIndPower` dijo que con d = 0.5 hacen falta unos 64 casos por grupo para 80% de potencia. Compruébalo: simula 5,000 pares de muestras de tamaño 64 (una con media 0 y otra con media 0.5, ambas con σ = 1; usa `generador = np.random.default_rng(401)`) y calcula la fracción de pruebas de Welch con *p* < 0.05. Repite con *n* = 20.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 3. Proporciones y variables categóricas (20 min).

* **Prueba *z* de dos proporciones** (`proportions_ztest(conteos, totales)`): ¿difiere la proporción de éxito entre dos grupos?
* **Chi cuadrada de independencia** (`stats.chi2_contingency(tabla)`): ¿están asociadas dos variables categóricas? Compara la tabla observada con la **esperada** si fueran independientes.
* **Supuesto clave:** frecuencias **esperadas** suficientemente grandes (en general ≥ 5 en cada celda). Si no se cumple, se agrupan categorías o se usa la prueba exacta de Fisher (`stats.fisher_exact`, para tablas 2×2).

* La siguiente celda comparará con la prueba *z* la tasa de supervivencia de mujeres y hombres en el Titanic.

In [ ]:
tabla_sexo = pd.crosstab(titanic["sex"], titanic["survived"])
sobrevivientes = tabla_sexo[1].to_numpy()
totales = tabla_sexo.sum(axis=1).to_numpy()
estadistico_z, valor_p_z = proportions_ztest(sobrevivientes, totales)
(sobrevivientes / totales).round(3), round(float(estadistico_z), 2), float(valor_p_z)

La diferencia es enorme (74.2% contra 18.9%) y su valor *p* es prácticamente 0. Para reportar la magnitud se usa la **diferencia de proporciones** con su IC (sesión 5) o el **riesgo relativo**: las mujeres sobrevivieron casi 4 veces más que los hombres.

**Ejemplo:** ¿están asociadas la clase y la supervivencia? Se revisa la tabla esperada antes de interpretar.

* La siguiente celda aplicará la chi cuadrada de independencia a `class` contra `survived` y mostrará la tabla esperada.

In [ ]:
tabla_clase = pd.crosstab(titanic["class"], titanic["survived"])
chi2 = stats.chi2_contingency(tabla_clase)
esperada = pd.DataFrame(chi2.expected_freq, index=tabla_clase.index, columns=tabla_clase.columns).round(1)
round(float(chi2.statistic), 1), int(chi2.dof), float(chi2.pvalue), esperada

Todas las frecuencias esperadas superan 5, así que la aproximación es válida. La chi cuadrada dice que hay asociación, pero no cuánta: el tamaño del efecto es la **V de Cramér** (0 = independencia, 1 = asociación perfecta), que se calcula con `stats.contingency.association(tabla, method="cramer")`.

| V de Cramér con 1 grado de libertad mínimo (orientativo) | 0.1 | 0.3 | 0.5 |
|---|---|---|---|
| Magnitud | Pequeña | Mediana | Grande |

### 🟢 Ejercicio 4 · Mujeres y hombres en tercera clase.

¿La ventaja de las mujeres se mantiene en tercera clase? Con los pasajeros de `class == "Third"`, compara la supervivencia de mujeres y hombres con la prueba *z* y calcula la V de Cramér de la tabla 2×2.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 4. Más de dos grupos (25 min).

Comparar tres grupos con tres pruebas *t* infla el error tipo I (sección 1). El procedimiento es:

1. **ANOVA de una vía** (`stats.f_oneway`): H0 = todas las medias son iguales. Supone independencia, normalidad aproximada y varianzas iguales.
2. **Levene** (`stats.levene`): ¿son iguales las varianzas? Si no, se usa el **ANOVA de Welch** (`anova_oneway(..., use_var="unequal")` de *statsmodels*).
3. Si se rechaza H0, **post hoc de Tukey** (`pairwise_tukeyhsd`) para saber **qué** pares difieren, controlando el error de las comparaciones múltiples.
4. Tamaño del efecto: **η²** = suma de cuadrados entre grupos / suma de cuadrados total (proporción de la variabilidad explicada por el grupo).

* La siguiente celda definirá `eta_cuadrada` a partir de los valores y las etiquetas de grupo.

In [ ]:
def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

**Ejemplo:** se usará `tip_pct` por día de la semana en `tips`, un caso donde **no** se espera una diferencia grande.

* La siguiente celda aplicará Levene, el ANOVA clásico, el ANOVA de Welch y Kruskal-Wallis (alternativa por rangos) a `tip_pct` por día, y calculará η².

In [ ]:
grupos_dia = [grupo["tip_pct"].to_numpy() for _, grupo in propinas.groupby("day")]
welch_dia = anova_oneway(propinas["tip_pct"], propinas["day"], use_var="unequal")
pd.DataFrame({
    "valor p": [stats.levene(*grupos_dia).pvalue, stats.f_oneway(*grupos_dia).pvalue,
                welch_dia.pvalue, stats.kruskal(*grupos_dia).pvalue],
}, index=["Levene", "ANOVA", "ANOVA de Welch", "Kruskal-Wallis"]).round(3), round(float(eta_cuadrada(propinas["tip_pct"], propinas["day"])), 3)

Ninguna prueba encuentra diferencias y η² es casi 0: el día explica apenas alrededor de 1% de la variabilidad de la propina porcentual. Como el ANOVA no rechaza, **no** se hace Tukey: buscar pares significativos después de un ANOVA no significativo es otra forma de *p-hacking*.

| η² (orientativo) | 0.01 | 0.06 | 0.14 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

**Consecuencia: significativo no es lo mismo que importante.** Con muestras enormes, cualquier diferencia mínima resulta significativa. Se simularán dos grupos de 100,000 personas cuyas medias difieren 0.45 puntos en una escala con σ = 15.

* La siguiente celda calculará el valor *p* de Welch y la d de Cohen de esa diferencia mínima.

In [ ]:
enorme_a = rng.normal(100, 15, size=100_000)
enorme_b = rng.normal(100.45, 15, size=100_000)
float(stats.ttest_ind(enorme_b, enorme_a, equal_var=False).pvalue), round(float(d_cohen(enorme_b, enorme_a)), 3)

El valor *p* es diminuto, pero la d es ≈ 0.03: un efecto **despreciable**. Reportar solo "*p* < 0.001" llevaría a creer que es un hallazgo importante. Por eso el tamaño del efecto y su IC son obligatorios en el informe.

### 🔴 Reto 2 · Doce pruebas a la vez.

Para cada especie y cada una de las cuatro medidas, compara machos y hembras con Welch (12 pruebas). Aplica las correcciones de Bonferroni y de Holm con `multipletests`. ¿Cuántas pruebas siguen siendo significativas? Agrega la d de Cohen de cada comparación: ¿cuál es la diferencia entre sexos más pequeña?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 5. Tamaño del efecto y ejercicio de la sesión (20 min).

| Situación | Prueba | Tamaño del efecto |
|---|---|---|
| Dos medias | Welch | d de Cohen (y la diferencia con su IC, en unidades originales) |
| Dos distribuciones asimétricas | Mann-Whitney o permutación de medianas | Diferencia de medianas con IC *bootstrap* |
| Dos proporciones | *z* de proporciones | Diferencia de proporciones o riesgo relativo |
| Dos categóricas | Chi cuadrada | V de Cramér |
| Tres o más medias | ANOVA (Welch si las varianzas difieren) + Tukey | η² |

**Plantilla para redactar:** "En estos datos, [grupo A] tuvo [medida] [mayor/menor] que [grupo B] ([valor A] contra [valor B]; diferencia [d], IC 95% [a, b]; [prueba], *p* [valor]; [tamaño del efecto] = [valor], un efecto [magnitud])." Sin verbos causales ("provoca", "hace que") cuando los datos son observacionales.

### 🟢 Ejercicio 5a · Masa por sexo en Adelie.

Con los resultados de la sección 2, redacta la conclusión de la comparación de masa entre machos y hembras Adelie en una o dos frases, con la plantilla.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 5b · Aleta por especie.

Compara `flipper_length_mm` entre las tres especies: revisa Levene, aplica el ANOVA (y el de Welch), calcula η² y, si corresponde, Tukey. Redacta la conclusión.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 5c · Clase y supervivencia.

Con la chi cuadrada de la sección 3, calcula la V de Cramér de `class` contra `survived` y redacta la conclusión sin lenguaje causal injustificado.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · Diagnóstico gráfico de supuestos.

Para el ejercicio 5b, dibuja un gráfico Q-Q de `flipper_length_mm` de cada especie (`stats.probplot(..., plot=ax)`) y un diagrama de caja por especie. ¿Las conclusiones del ANOVA dependen de algún supuesto dudoso?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La lógica del contraste, qué **no** significa un valor *p*, la potencia y por qué las comparaciones múltiples producen falsos hallazgos.
* Dos grupos: Welch por defecto (Student falla con varianzas y tamaños distintos), Mann-Whitney y permutación.
* Proporciones (*z*) y categóricas (chi cuadrada, revisando las frecuencias esperadas).
* Más de dos grupos: Levene, ANOVA clásico y de Welch, Tukey solo si el ANOVA rechaza.
* Tamaño del efecto: d de Cohen, V de Cramér y η²; con *n* enorme, significativo no implica importante.

**Práctica (sin calificación, en Classroom):** completar los tres incisos del ejercicio 5 y aplicar al menos una prueba con su tamaño del efecto al *dataset* del proyecto.

**Próxima sesión (mié 28 oct):** EDA completo paso a paso con `mpg` y arranque del proyecto (`07_eda_completo.ipynb`).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>